# CatalogueIQ — RAG-Powered Product Intelligence Assistant

**ShopSmart India | Capstone Level 2**

CatalogueIQ is a Retrieval-Augmented Generation (RAG) assistant for ShopSmart India. It combines a product catalogue with returns and refunds policy, seller onboarding guidance, category attribute documentation, buyer FAQs, and product review summaries. The pipeline supports grounded product lookup, policy eligibility, comparative recommendations, seller policy questions, and multi-hop reasoning with source citations.

The implementation includes mixed-format ingestion, file-specific chunking, OpenAI embeddings, FAISS vector search, query expansion, metadata filtering, hybrid BM25 retrieval, SQLite filtering, five-turn conversation memory, three personas, session personalization, and RAGAS evaluation.

## 1. Project Overview

### System flow

**CSV / Markdown / HTML → Loaders → Structure-aware chunks → Embeddings → FAISS + BM25 → Query expansion + metadata filtering → Grounded generation → Citations**

### Knowledge base

- Product catalogue and review summaries
- Returns and refunds policy
- Big Billion Days promotional terms
- Seller onboarding guide
- Category taxonomy and attribute guides
- Buyer FAQ

### Query coverage

The assistant is designed for product factual lookup, policy and eligibility questions, comparative recommendations, seller policy lookup, and multi-hop reasoning.

## 2. Environment Setup

The notebook uses OpenAI for embeddings and generation, FAISS for dense retrieval, BM25 for lexical retrieval, SQLite for structured filtering, Gradio for the three-persona interface, and RAGAS for evaluation.

In [23]:
%pip install -q -U pandas numpy scikit-learn faiss-cpu rank-bm25 beautifulsoup4 lxml gradio openai python-dotenv ragas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from dotenv import load_dotenv
import os, getpass

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("OpenAI API key (hidden input): ").strip()
    if OPENAI_API_KEY:
        os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

EMBED_MODEL_NAME = "text-embedding-3-small"
GENERATION_MODEL = os.getenv("CATALOGUEIQ_OPENAI_MODEL", "gpt-4o-mini")

print("OpenAI embeddings:", EMBED_MODEL_NAME)
print("Generation model:", GENERATION_MODEL)
print("API key configured:", bool(OPENAI_API_KEY))


OpenAI embeddings: text-embedding-3-small
Generation model: gpt-4o-mini
API key configured: True


In [25]:
import os, re, json, math, sqlite3, hashlib, time, html
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

DATA_DIR = Path("shopsmart_data")
CACHE_DIR = Path("shopsmart_cache")
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)
EMBED_MODEL_NAME = "text-embedding-3-small"

print("Data directory:", DATA_DIR.resolve())
print("Cache directory:", CACHE_DIR.resolve())

Data directory: C:\data\Python-Trainings\Python-Trainings\Frontier-Engineer\week1\shopsmart_data
Cache directory: C:\data\Python-Trainings\Python-Trainings\Frontier-Engineer\week1\shopsmart_cache


## 3. Build the ShopSmart India knowledge base

The capstone allows teams to construct/source the fictional ShopSmart data. To keep this notebook reproducible, the dataset is generated deterministically with realistic Indian e-commerce fields.

We generate:

- 500 products across Electronics, Fashion, Home & Kitchen, Beauty, and Books
- returns/refunds policy
- promotional terms for the Big Billion Days example
- seller onboarding guide
- category taxonomy
- 5 category-specific attribute guides
- Buyer FAQ with 50 questions
- product review summaries

In [26]:
from itertools import cycle

categories = {
    "Electronics": {
        "brands": ["boAt", "Sony", "JBL", "Noise", "OnePlus", "Samsung", "realme", "Philips", "HP", "Lenovo"],
        "types": ["Wireless Earbuds", "Smartwatch", "Bluetooth Speaker", "Headphones", "Power Bank", "Keyboard", "Mouse", "Monitor", "Webcam", "Soundbar"],
    },
    "Fashion": {
        "brands": ["Roadster", "Wrogn", "Aurelia", "Libas", "Flying Machine", "Allen Solly", "Van Heusen", "Puma", "Campus", "Mast & Harbour"],
        "types": ["T-Shirt", "Jeans", "Kurta", "Sneakers", "Hoodie", "Dress", "Shirt", "Jacket", "Sandals", "Backpack"],
    },
    "Home & Kitchen": {
        "brands": ["Prestige", "Pigeon", "Borosil", "Milton", "Havells", "Philips", "Butterfly", "Cello", "Wakefit", "Amazon Basics"],
        "types": ["Mixer Grinder", "Air Fryer", "Cookware Set", "Water Bottle", "Bedsheet", "Table Lamp", "Storage Box", "Electric Kettle", "Pressure Cooker", "Non-stick Pan"],
    },
    "Beauty": {
        "brands": ["Lakme", "Mamaearth", "Minimalist", "Plum", "Maybelline", "The Derma Co", "Nykaa", "Biotique", "WOW Skin Science", "Nivea"],
        "types": ["Face Wash", "Sunscreen", "Moisturizer", "Serum", "Shampoo", "Conditioner", "Lipstick", "Face Mask", "Body Lotion", "Perfume"],
    },
    "Books": {
        "brands": ["Penguin Random House", "HarperCollins", "Rupa", "Bloomsbury", "S. Chand", "Pearson", "Oxford", "McGraw Hill", "Hachette", "Westland"],
        "types": ["Fiction Novel", "Business Book", "Self Help", "Computer Science", "Data Science", "Biography", "History", "Exam Prep", "Finance", "Psychology"],
    },
}

colors = ["Black", "Blue", "White", "Grey", "Red", "Green", "Beige"]

products = []
product_id = 1001

# Hand-crafted anchor products used by demos/tests.
anchors = [
    dict(product_id="P1001", category="Electronics", product_type="Wireless Earbuds", name="boAt Airdopes 141", brand="boAt", price=1499, rating=4.2, battery_hours=42, anc=False, warranty_months=12, seller_id="S001", return_days=7, verified_seller=True, tags="earbuds, earphones, TWS, in-ear, wireless"),
    dict(product_id="P1002", category="Electronics", product_type="Wireless Earbuds", name="Sony WF-C700N", brand="Sony", price=2799, rating=4.5, battery_hours=30, anc=True, warranty_months=12, seller_id="S002", return_days=7, verified_seller=True, tags="earbuds, earphones, TWS, ANC, noise cancellation"),
    dict(product_id="P1003", category="Electronics", product_type="Wireless Earbuds", name="JBL Wave Beam", brand="JBL", price=2499, rating=4.4, battery_hours=32, anc=False, warranty_months=12, seller_id="S003", return_days=7, verified_seller=True, tags="earbuds, earphones, TWS, deep bass"),
    dict(product_id="P1004", category="Electronics", product_type="Wireless Earbuds", name="OnePlus Nord Buds 2", brand="OnePlus", price=2299, rating=4.3, battery_hours=36, anc=True, warranty_months=12, seller_id="S004", return_days=7, verified_seller=True, tags="earbuds, earphones, TWS, ANC, noise cancellation"),
    dict(product_id="P1005", category="Electronics", product_type="Wireless Earbuds", name="realme Buds T300", brand="realme", price=1999, rating=4.3, battery_hours=40, anc=True, warranty_months=12, seller_id="S005", return_days=7, verified_seller=True, tags="earbuds, earphones, TWS, ANC, noise cancellation"),
    dict(product_id="P1010", category="Electronics", product_type="Smartwatch", name="Noise ColorFit Ultra 3", brand="Noise", price=3499, rating=4.2, battery_hours=192, anc=False, warranty_months=12, seller_id="S006", return_days=7, verified_seller=False, tags="smartwatch, watch, fitness, AMOLED"),
]
products.extend(anchors)
product_id = 1011

for category, spec in categories.items():
    existing = [p for p in products if p["category"] == category]
    next_count = 100 - len(existing)
    for i in range(next_count):
        brand = spec["brands"][i % len(spec["brands"])]; ptype = spec["types"][i % len(spec["types"])];
        idx = len(existing) + i + 1
        pid = f"P{product_id}"
        price = int(rng.integers(299, 8999) // 50 * 50 + 49)
        rating = round(float(rng.choice([3.7, 3.9, 4.0, 4.1, 4.2, 4.3, 4.4, 4.5, 4.6])), 1)
        battery = int(rng.integers(6, 55)) if category == "Electronics" else None
        anc = bool(rng.integers(0, 2)) if ptype in ["Wireless Earbuds", "Headphones"] else False
        if ptype == "Wireless Earbuds":
            price = int(rng.choice([999, 1299, 1499, 1699, 1899, 2199, 2499, 2799, 2999]))
            battery = int(rng.integers(20, 50))
        warranty = int(rng.choice([6, 12, 18, 24]))
        seller_id = f"S{((product_id - 1000) % 25) + 1:03d}"
        returns = 7 if category == "Electronics" else 10 if category == "Fashion" else 7 if category == "Beauty" else 10
        tag_terms = [ptype.lower(), brand.lower()]
        if ptype == "Wireless Earbuds": tag_terms += ["earbuds", "earphones", "TWS"]
        name = f"{brand} {ptype} {idx}"
        products.append(dict(
            product_id=pid, category=category, product_type=ptype, name=name, brand=brand, price=price,
            rating=rating, battery_hours=battery, anc=anc, warranty_months=warranty, seller_id=seller_id,
            return_days=returns, verified_seller=bool((product_id % 4) != 0), tags=", ".join(tag_terms)
        ))
        product_id += 1

catalogue_df = pd.DataFrame(products)
catalogue_df["specs"] = catalogue_df.apply(lambda r: json.dumps({
    "product_type": r.product_type, "battery_hours": r.battery_hours, "anc": r.anc,
    "warranty_months": r.warranty_months, "return_days": r.return_days,
    "verified_seller": r.verified_seller, "tags": r.tags
}), axis=1)

catalogue_path = DATA_DIR / "product_catalogue.csv"
catalogue_df.to_csv(catalogue_path, index=False)

print(catalogue_df.shape)
print(catalogue_df.groupby("category").size())
print(catalogue_df.head(8)[["product_id", "name", "category", "price", "rating", "seller_id"]])

(500, 15)
category
Beauty            100
Books             100
Electronics       100
Fashion           100
Home & Kitchen    100
dtype: int64
  product_id                     name     category  price  rating seller_id
0      P1001        boAt Airdopes 141  Electronics   1499     4.2      S001
1      P1002            Sony WF-C700N  Electronics   2799     4.5      S002
2      P1003            JBL Wave Beam  Electronics   2499     4.4      S003
3      P1004      OnePlus Nord Buds 2  Electronics   2299     4.3      S004
4      P1005         realme Buds T300  Electronics   1999     4.3      S005
5      P1010   Noise ColorFit Ultra 3  Electronics   3499     4.2      S006
6      P1011  boAt Wireless Earbuds 7  Electronics   1699     4.4      S012
7      P1012        Sony Smartwatch 8  Electronics   6399     3.9      S013


In [27]:
category_taxonomy = """# ShopSmart India Category Taxonomy

## Electronics
- Audio: Wireless Earbuds, Headphones, Speakers, Soundbars
- Wearables: Smartwatches
- Computing: Keyboards, Mice, Monitors, Webcams
- Power: Power Banks

Tagging conventions: use `audio`, `wireless`, `TWS`, `ANC`, `wearable`, `display`, and `battery` where applicable.

## Fashion
- Apparel: T-Shirts, Shirts, Jeans, Kurtas, Dresses, Hoodies, Jackets
- Footwear: Sneakers, Sandals
- Accessories: Backpacks

## Home & Kitchen
- Appliances: Mixer Grinders, Air Fryers, Kettles
- Cookware: Cookware Sets, Pressure Cookers, Non-stick Pans
- Home Utility: Bottles, Bedsheets, Lamps, Storage

## Beauty
- Skin: Face Wash, Sunscreen, Moisturizer, Serum, Masks
- Hair: Shampoo, Conditioner
- Makeup & Fragrance: Lipstick, Perfume

## Books
- Fiction
- Business & Finance
- Technology & Computer Science
- Self Help & Psychology
- Biography & History
- Exam Preparation
"""
(DATA_DIR / "category_taxonomy.md").write_text(category_taxonomy, encoding="utf-8")

returns_policy = """# ShopSmart India Returns & Refunds Policy

## Standard return windows
Most eligible Electronics and Beauty items can be returned within 7 days of delivery. Eligible Fashion and Books items can be returned within 10 days. The item must be unused, undamaged, and include original packaging, accessories, tags, and invoice where applicable.

## Sale and promotional purchases
During a promotional event such as Big Billion Days, an item keeps its normal category return window unless the product listing or promotion explicitly marks it as non-returnable. A discounted price by itself does not make an otherwise eligible product non-returnable.

## Non-returnable exceptions
Personalised products, intimate-use products, opened hygiene-sealed cosmetics, downloadable/digital goods, and products explicitly labelled non-returnable are not eligible for change-of-mind returns. Damaged or incorrect deliveries are handled through buyer protection even when a product is otherwise non-returnable.

## Refund timelines
After a return is received and passes inspection, refunds are generally initiated within 2 business days. Bank/card processing can take 3-7 additional business days. Wallet refunds are usually visible within 24 hours after initiation.

## Warranty
Warranty is separate from the return window. If an item develops a functional defect after the return period, the buyer may use the manufacturer warranty when a warranty is listed on the product page. Warranty coverage normally applies to manufacturing defects and excludes accidental, liquid, cosmetic, and unauthorised-repair damage.

## Third-party sellers
For marketplace sellers, the seller is responsible for fulfilling the order and responding to eligible service issues. ShopSmart buyer protection can assist with delivery failures, wrong or damaged items, and eligible disputes. Manufacturer warranty claims may still be handled by the manufacturer or authorised service network depending on the product.

## Buyer protection
For a product that stops working after the standard return period, first verify warranty status. If the product is still under warranty, submit the invoice and defect details through the warranty path. If the seller did not disclose required information, or the item was materially different, damaged, or counterfeit, Buyer Protection may provide a dispute route.
"""
(DATA_DIR / "returns_refunds_policy.md").write_text(returns_policy, encoding="utf-8")

promotions = """# ShopSmart Big Billion Days Promotional Terms

## Return treatment for sale items
Discounted items retain the category's normal return window unless the listing explicitly displays a non-returnable exception. A sale price alone is not a reason to reject an otherwise eligible return.

## Promotional exclusions
Products marked final sale, personalised, hygiene-sealed and opened, digital goods, or otherwise marked non-returnable remain excluded.
"""
(DATA_DIR / "big_billion_days_terms.md").write_text(promotions, encoding="utf-8")

seller_guide = """# Seller Onboarding Guide

## Account setup
Create a seller account, verify the business contact details, submit tax and payout information, and complete KYC as required by ShopSmart India.

## Product listing requirements
Every listing must include a clear product name, brand, category, key specifications, price, tax-inclusive selling price where applicable, stock status, warranty information, return eligibility, and accurate seller ID metadata.

## Electronics image requirements
Electronics listings require at least 3 clear images: a front/primary product image, an angled or side view, and a packaging/accessories view. Images should use a plain or uncluttered background, show the actual product, avoid misleading watermarks, and meet ShopSmart quality/compression limits. Do not upload screenshots of competitor listings.

## Fashion image requirements
Fashion listings should show the garment or item clearly, include front and back views where relevant, and show size/fit information. Images should not hide important design details.

## Image quality rules
Use original or licensed images. Avoid heavy text overlays, deceptive before/after claims, irrelevant props, and images that materially differ from the sold product.

## Pricing rules
Displayed pricing must match the seller's submitted price. Promotional discounts must be supported by a valid base price and cannot be used to misrepresent a permanent discount.

## Compliance checklist
Confirm product authenticity, brand authorisation where required, accurate specifications, safe packaging, correct tax information, warranty details, and policy-compliant return settings before publishing.
"""
(DATA_DIR / "seller_onboarding_guide.md").write_text(seller_guide, encoding="utf-8")

attribute_guides = {
    "electronics_attributes.md": """# Electronics Attribute Guide

## Wireless Earbuds
Battery life is the stated playback time, usually in hours. ANC means active noise cancellation. TWS means true wireless stereo.

## Smartwatch
Battery life should be represented in hours or days depending on the listing. Key attributes include display type, sensors, calling support, and water resistance.

## Audio comparisons
For earbuds, compare price, battery hours, ANC, rating, and warranty. Do not infer ANC from brand or price.
""",
    "fashion_attributes.md": """# Fashion Attribute Guide

## Apparel
Important attributes include fabric, size, fit, colour, wash-care instructions, and occasion.

## Footwear
Use size, material, sole type, closure, and fit.

## Recommendation rule
Do not recommend a size unless the listing contains usable sizing information.
""",
    "home_kitchen_attributes.md": """# Home & Kitchen Attribute Guide

## Appliances
Compare capacity, power, warranty, and safety features.

## Cookware
Compare material, induction compatibility, capacity, and care instructions.
""",
    "beauty_attributes.md": """# Beauty Attribute Guide

## Skin care
Use skin type, key ingredient, SPF where relevant, volume, and suitability claims supplied by the listing.

## Non-returnable hygiene items
Opened hygiene-sealed products can be excluded from change-of-mind returns under the returns policy.
""",
    "books_attributes.md": """# Books Attribute Guide

## Standard fields
Use author, publisher, edition, format, page count, language, rating, and price.

## Recommendations
Prefer rating and topic fit; do not invent plot details not present in the catalogue or review summary.
""",
}
for name, text in attribute_guides.items():
    (DATA_DIR / name).write_text(text, encoding="utf-8")

faq_items = [
    ("Can I cancel an order?", "Cancellation is allowed before dispatch when the order page provides the cancel option."),
    ("How long does delivery take?", "Estimated delivery depends on seller location, destination, and shipping method; use the order page for the current estimate."),
    ("How do I track an order?", "Open Orders, select the order, and use the tracking link when available."),
    ("How do refunds work?", "Refunds are initiated after return inspection and can take additional banking time."),
    ("Can I return a sale item?", "Sale items keep the normal category return window unless the listing explicitly marks the item non-returnable."),
    ("What is buyer protection?", "Buyer Protection provides a dispute route for eligible delivery, wrong-item, damaged, counterfeit, or materially different issues."),
    ("Are personalised items returnable?", "Personalised items are generally not eligible for change-of-mind returns."),
    ("How do I contact a seller?", "Use the order or product page seller contact workflow when available."),
    ("Can I change my address?", "Address changes depend on the order status; use the order page before dispatch."),
    ("What payment methods are supported?", "Available payment methods are shown during checkout and can vary by order."),
    ("How do I update my mobile number?", "Update contact details from the account profile when the option is available."),
    ("Can I use multiple payment methods?", "Checkout shows whether split payments are available for the current order."),
    ("What happens if my package is delayed?", "Check the current tracking estimate and contact support if the promised window has passed."),
    ("What if I receive the wrong product?", "Wrong-item deliveries can be handled through the return or Buyer Protection process."),
    ("What if my package arrives damaged?", "Report the damage promptly and use the eligible return or Buyer Protection route."),
    ("Can I return an opened cosmetic?", "Opened hygiene-sealed cosmetics can be excluded from change-of-mind returns."),
    ("Are digital goods returnable?", "Downloadable or digital goods are generally non-returnable."),
    ("When does the return clock start?", "The category return window is measured from delivery according to the return policy."),
    ("Do return windows differ by category?", "Yes. Electronics/Beauty generally use 7 days, while Fashion/Books generally use 10 days under this catalogue policy."),
    ("Can I return an item without original packaging?", "Eligible returns normally require original packaging and applicable accessories."),
    ("Do I need the invoice for a return?", "The policy says the invoice should be included where applicable."),
    ("How long does a refund take after inspection?", "Refund initiation is generally within 2 business days after successful inspection."),
    ("How long can a card refund take to appear?", "Bank/card processing can take 3-7 additional business days after initiation."),
    ("How fast are wallet refunds?", "Wallet refunds are usually visible within 24 hours after initiation."),
    ("What does warranty cover?", "Warranty normally covers manufacturing defects when a warranty is listed."),
    ("Does warranty cover accidental damage?", "Accidental, liquid, cosmetic, and unauthorised-repair damage are generally excluded."),
    ("Can I use warranty after the return period?", "Yes. Warranty is separate from the return window and may apply after the return period."),
    ("What should I submit for a warranty claim?", "Submit the invoice and defect details through the warranty path."),
    ("Who handles manufacturer warranty?", "The manufacturer or authorised service network may handle warranty claims depending on the product."),
    ("What if a seller is unresponsive?", "Eligible seller/order disputes can be raised through Buyer Protection; warranty claims can use the warranty path."),
    ("How can I verify a seller?", "The catalogue exposes a verified-seller field for products that include that metadata."),
    ("Can a sale price remove my return rights?", "A sale price alone does not remove the normal category return window."),
    ("What if a listing says final sale?", "A product explicitly marked non-returnable can be excluded from change-of-mind returns."),
    ("Are personalised gifts returnable?", "Personalised items are generally excluded from change-of-mind returns."),
    ("Can I return a damaged non-returnable item?", "Damaged or incorrect deliveries can still be handled through Buyer Protection."),
    ("Can I change an order after dispatch?", "Changes depend on order status; after dispatch the available options may be limited."),
    ("How do I find my order invoice?", "Open the relevant order and use the invoice option when provided."),
    ("Can I request a replacement instead of refund?", "The available resolution depends on the product and order workflow shown by ShopSmart."),
    ("What if my refund is missing?", "Check the stated bank/card or wallet processing window before escalating."),
    ("Does ShopSmart guarantee every product is authentic?", "Buyer Protection provides a dispute route for eligible counterfeit or materially different products."),
    ("What if a product is not as described?", "A materially different product can be handled through Buyer Protection."),
    ("How do I report a counterfeit product?", "Use the eligible Buyer Protection/dispute route and provide order evidence."),
    ("Can a seller set a custom return window?", "The listing must use ShopSmart-compliant return settings; category policy still governs standard eligibility."),
    ("What happens when a return is rejected?", "Return eligibility depends on policy conditions and inspection findings."),
    ("Can I return a product just because I changed my mind?", "Only if the product is eligible under the category return policy and is in the required condition."),
    ("Do tags matter for Fashion returns?", "Yes. Eligible Fashion returns normally require original tags and applicable packaging."),
    ("Are opened hygiene products eligible?", "Opened hygiene-sealed products can be non-returnable for change-of-mind reasons."),
    ("Can Buyer Protection help outside the return window?", "It can provide a route for eligible delivery, damage, wrong-item, counterfeit, or materially different disputes."),
    ("Where can I see current return eligibility?", "Check the product listing and order page for the applicable return status and policy information."),
    ("What if the product stops working after 45 days?", "The standard return period may have ended; check the listed warranty and use Buyer Protection for eligible seller/order disputes."),
]
faq_html = "<html><body><h1>ShopSmart India Buyer FAQ</h1>" + "".join(f"<section><h2>FAQ {i+1}</h2><p><strong>Q:</strong> {q}</p><p><strong>A:</strong> {a}</p></section>" for i,(q,a) in enumerate(faq_items)) + "</body></html>"
(DATA_DIR / "buyer_faq.html").write_text(faq_html, encoding="utf-8")

reviews = []
for _, r in catalogue_df.sort_values("rating", ascending=False).groupby("category").head(10).iterrows():
    theme = "long battery and value" if r.product_type == "Wireless Earbuds" else "good build quality and value" if r.category != "Books" else "clear writing and useful content"
    reviews.append({"product_id": r.product_id, "review_summary": f"Review themes mention {theme}; average rating in catalogue is {r.rating}."})
review_df = pd.DataFrame(reviews)
review_df.to_csv(DATA_DIR / "product_review_summaries.csv", index=False)

print("Created files:")
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)

Created files:
 - beauty_attributes.md
 - big_billion_days_terms.md
 - books_attributes.md
 - buyer_faq.html
 - category_taxonomy.md
 - electronics_attributes.md
 - fashion_attributes.md
 - home_kitchen_attributes.md
 - product_catalogue.csv
 - product_review_summaries.csv
 - returns_refunds_policy.md
 - seller_onboarding_guide.md


## 4. Loaders: CSV, Markdown, and HTML

A document is converted to a common chunk structure with text plus metadata. Metadata is later used for filtering and citations.

In [28]:
@dataclass
class DocumentChunk:
    text: str
    source: str
    section: str
    doc_type: str
    metadata: Dict[str, Any] = field(default_factory=dict)


def load_csv_products(path: Path) -> List[DocumentChunk]:
    df = pd.read_csv(path)
    chunks = []
    for _, r in df.iterrows():
        text = (
            f"Product ID: {r.product_id}\n"
            f"Name: {r['name']}\n"
            f"Brand: {r.brand}\n"
            f"Category: {r.category}\n"
            f"Product type: {r.product_type}\n"
            f"Price: ₹{int(r.price)}\n"
            f"Rating: {r.rating}/5\n"
            f"Battery life: {r.battery_hours if pd.notna(r.battery_hours) else 'N/A'} hours\n"
            f"ANC: {'Yes' if r.anc else 'No'}\n"
            f"Warranty: {int(r.warranty_months)} months\n"
            f"Return window: {int(r.return_days)} days\n"
            f"Seller ID: {r.seller_id}\n"
            f"Verified seller: {'Yes' if r.verified_seller else 'No'}\n"
            f"Tags: {r.tags}"
        )
        chunks.append(DocumentChunk(text, "product_catalogue.csv", r.product_id, "catalogue", {
            "product_id": r.product_id, "category": r.category, "price": float(r.price),
            "rating": float(r.rating), "brand": r.brand, "product_type": r.product_type,
            "seller_id": r.seller_id, "verified_seller": bool(r.verified_seller)
        }))
    return chunks


def load_markdown(path: Path) -> List[DocumentChunk]:
    text = path.read_text(encoding="utf-8")
    chunks = []
    current = "Document"
    buf = []
    for line in text.splitlines():
        if line.startswith("#"):
            if buf and " ".join(buf).strip():
                chunks.append(DocumentChunk(" ".join(buf).strip(), path.name, current, "markdown", {}))
            current = re.sub(r"^#+\s*", "", line).strip()
            buf = []
        elif line.strip():
            buf.append(line.strip())
    if buf:
        chunks.append(DocumentChunk(" ".join(buf).strip(), path.name, current, "markdown", {}))
    return chunks


def load_html_faq(path: Path) -> List[DocumentChunk]:
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "lxml")
    chunks = []
    for section in soup.find_all("section"):
        heading = section.find("h2").get_text(" ", strip=True) if section.find("h2") else "FAQ"
        text = section.get_text(" ", strip=True)
        chunks.append(DocumentChunk(text, path.name, heading, "html", {}))
    return chunks


def ingest_all(data_dir: Path) -> List[DocumentChunk]:
    chunks = []
    chunks += load_csv_products(data_dir / "product_catalogue.csv")
    for p in [data_dir / "returns_refunds_policy.md", data_dir / "big_billion_days_terms.md", data_dir / "seller_onboarding_guide.md", data_dir / "category_taxonomy.md"] + sorted(data_dir.glob("*_attributes.md")):
        chunks += load_markdown(p)
    chunks += load_html_faq(data_dir / "buyer_faq.html")
    # review CSV is included as a separate document type
    for _, r in pd.read_csv(data_dir / "product_review_summaries.csv").iterrows():
        chunks.append(DocumentChunk(f"Product ID: {r.product_id}\n{r.review_summary}", "product_review_summaries.csv", r.product_id, "reviews", {"product_id": r.product_id}))
    return chunks

raw_chunks = ingest_all(DATA_DIR)
print("Raw chunks:", len(raw_chunks))
print(Counter(c.doc_type for c in raw_chunks))

Raw chunks: 633
Counter({'catalogue': 500, 'html': 50, 'reviews': 50, 'markdown': 33})


## 5. Two chunking strategies (required experiment)

**Strategy A — fixed-size:** split every document into character windows. This is simple, but it can split a product row or a policy heading in the middle.

**Strategy B — structure-aware:** keep each product record as one natural chunk; split Markdown/HTML by heading/FAQ section with a safe maximum length; keep review rows separate.

The second strategy is chosen because structured product data has a natural row-level semantic boundary, while policy prose has heading/paragraph boundaries.

In [29]:
def fixed_size_chunks(raw: List[DocumentChunk], size: int = 180, overlap: int = 30) -> List[DocumentChunk]:
    out = []
    for d in raw:
        text = d.text
        if len(text) <= size:
            out.append(d); continue
        start = 0; idx = 0
        while start < len(text):
            part = text[start:start+size]
            out.append(DocumentChunk(part, d.source, f"{d.section} [fixed-{idx}]", d.doc_type, dict(d.metadata)))
            idx += 1
            start += size - overlap
    return out


def split_long_text(text: str, max_chars: int = 700) -> List[str]:
    paras = [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]
    pieces, buf = [], ""
    for p in paras:
        if len(buf) + len(p) + 1 <= max_chars:
            buf = (buf + " " + p).strip()
        else:
            if buf: pieces.append(buf)
            buf = p
    if buf: pieces.append(buf)
    return pieces


def structure_aware_chunks(raw: List[DocumentChunk]) -> List[DocumentChunk]:
    out = []
    for d in raw:
        if d.doc_type in {"catalogue", "reviews", "html"}:
            out.append(d)
        else:
            for i, piece in enumerate(split_long_text(d.text)):
                out.append(DocumentChunk(piece, d.source, d.section if i == 0 else f"{d.section} [{i+1}]", d.doc_type, dict(d.metadata)))
    return out

fixed_chunks = fixed_size_chunks(raw_chunks)
smart_chunks = structure_aware_chunks(raw_chunks)

chunk_compare = pd.DataFrame([
    {"strategy": "Fixed-size", "total_chunks": len(fixed_chunks), "catalogue_chunks": sum(c.doc_type == "catalogue" for c in fixed_chunks), "avg_chars": np.mean([len(c.text) for c in fixed_chunks])},
    {"strategy": "Structure-aware", "total_chunks": len(smart_chunks), "catalogue_chunks": sum(c.doc_type == "catalogue" for c in smart_chunks), "avg_chars": np.mean([len(c.text) for c in smart_chunks])},
])
print(chunk_compare)
all_text = " ".join(c.text for c in smart_chunks)
approx_tokens = max(1, len(all_text) // 4)
print(f"Approximate structure-aware token count (chars/4): {approx_tokens:,}")
print("\nExample product chunk (structure-aware):\n", smart_chunks[0].text)

          strategy  total_chunks  catalogue_chunks   avg_chars
0       Fixed-size          1184              1032  142.507601
1  Structure-aware           633               500  241.627172
Approximate structure-aware token count (chars/4): 38,395

Example product chunk (structure-aware):
 Product ID: P1001
Name: boAt Airdopes 141
Brand: boAt
Category: Electronics
Product type: Wireless Earbuds
Price: ₹1499
Rating: 4.2/5
Battery life: 42.0 hours
ANC: No
Warranty: 12 months
Return window: 7 days
Seller ID: S001
Verified seller: Yes
Tags: earbuds, earphones, TWS, in-ear, wireless


### Chunk inspection

This cell prints the retrieved-style chunks for the hardest class: comparative recommendation. At this point there is no generation yet; only retrieval quality matters.

In [30]:
# Lightweight lexical inspection before embeddings.
query = "best wireless earbuds under 2000 rupees with ANC and long battery life"
keywords = ["earbuds", "ANC", "battery", "2000"]
preview = []
for c in smart_chunks:
    score = sum(k.lower() in c.text.lower() for k in keywords)
    if score >= 2 and c.doc_type == "catalogue":
        preview.append((score, c))
for score, c in sorted(preview, key=lambda x: x[0], reverse=True)[:8]:
    print(f"score={score} | {c.section} | {c.text.replace(chr(10), ' | ')}\n")

score=3 | P1001 | Product ID: P1001 | Name: boAt Airdopes 141 | Brand: boAt | Category: Electronics | Product type: Wireless Earbuds | Price: ₹1499 | Rating: 4.2/5 | Battery life: 42.0 hours | ANC: No | Warranty: 12 months | Return window: 7 days | Seller ID: S001 | Verified seller: Yes | Tags: earbuds, earphones, TWS, in-ear, wireless

score=3 | P1002 | Product ID: P1002 | Name: Sony WF-C700N | Brand: Sony | Category: Electronics | Product type: Wireless Earbuds | Price: ₹2799 | Rating: 4.5/5 | Battery life: 30.0 hours | ANC: Yes | Warranty: 12 months | Return window: 7 days | Seller ID: S002 | Verified seller: Yes | Tags: earbuds, earphones, TWS, ANC, noise cancellation

score=3 | P1003 | Product ID: P1003 | Name: JBL Wave Beam | Brand: JBL | Category: Electronics | Product type: Wireless Earbuds | Price: ₹2499 | Rating: 4.4/5 | Battery life: 32.0 hours | ANC: No | Warranty: 12 months | Return window: 7 days | Seller ID: S003 | Verified seller: Yes | Tags: earbuds, earphones, TWS, de

## 6. Embeddings + FAISS Index + Embedding Cache

The final embedding model is OpenAI `text-embedding-3-small`, which is explicitly permitted by the capstone specification. Embeddings are cached by content hash so unchanged chunks do not need to be embedded again. FAISS `IndexFlatIP` stores normalized dense vectors for cosine-similarity search.


In [31]:
import hashlib
import numpy as np

class EmbeddingStore:
    def __init__(self, model_name: str, cache_path: Path):
        self.model_name = model_name
        self.cache_path = cache_path
        self.index = None
        self.chunks: List[DocumentChunk] = []
        self.embeddings = None
        self.client = None
        if OPENAI_API_KEY:
            from openai import OpenAI
            self.client = OpenAI(api_key=OPENAI_API_KEY)

    def _cache_file(self, texts: List[str]) -> Path:
        h = hashlib.md5((self.model_name + "\n" + "\n".join(texts)).encode("utf-8")).hexdigest()
        return self.cache_path / f"embeddings_{h}.npy"

    def _embed_openai(self, texts: List[str], batch_size: int = 100) -> np.ndarray:
        vectors = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            response = self.client.embeddings.create(model=self.model_name, input=batch)
            vectors.extend([item.embedding for item in response.data])
        arr = np.asarray(vectors, dtype="float32")
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        return arr / np.clip(norms, 1e-12, None)

    def embed_query(self, query: str) -> np.ndarray:
        if not self.client:
            raise RuntimeError("OPENAI_API_KEY is required for OpenAI embeddings.")
        return self._embed_openai([query])[0]

    def build(self, chunks: List[DocumentChunk]):
        texts = [c.text for c in chunks]
        cache_file = self._cache_file(texts)
        if cache_file.exists():
            emb = np.load(cache_file)
        else:
            if not self.client:
                print("OPENAI_API_KEY is not configured. Embeddings are not built in this session.")
                self.chunks = chunks
                return None
            emb = self._embed_openai(texts)
            np.save(cache_file, emb)
        emb = np.asarray(emb, dtype="float32")
        self.embeddings = emb
        self.chunks = chunks
        try:
            import faiss
            self.index = faiss.IndexFlatIP(emb.shape[1])
            self.index.add(emb)
            print(f"FAISS vectors: {self.index.ntotal} | dimension: {emb.shape[1]}")
        except ImportError:
            self.index = None
            print("FAISS is unavailable; dense retrieval will use the cached embedding matrix.")
        return emb

smart_store = EmbeddingStore(EMBED_MODEL_NAME, CACHE_DIR)
smart_embeddings = smart_store.build(smart_chunks)


FAISS vectors: 633 | dimension: 1536


## 7. SQLite product index (stretch goal)

SQLite handles exact numeric filters better than vector search: price range, rating, category, and brand. We use it to narrow the candidate product pool before semantic retrieval.

In [32]:
sqlite_path = CACHE_DIR / "products.db"
con = sqlite3.connect(sqlite_path)
catalogue_df.to_sql("products", con, if_exists="replace", index=False)
con.execute("CREATE INDEX IF NOT EXISTS idx_category ON products(category)")
con.execute("CREATE INDEX IF NOT EXISTS idx_price ON products(price)")
con.execute("CREATE INDEX IF NOT EXISTS idx_rating ON products(rating)")
con.commit()


def sql_product_ids(category: Optional[str] = None, min_price: Optional[float] = None, max_price: Optional[float] = None) -> set:
    q = "SELECT product_id FROM products WHERE 1=1"
    args = []
    if category:
        q += " AND category = ?"; args.append(category)
    if min_price is not None:
        q += " AND price >= ?"; args.append(min_price)
    if max_price is not None:
        q += " AND price <= ?"; args.append(max_price)
    return {r[0] for r in con.execute(q, args).fetchall()}

print("Products in SQLite:", con.execute("SELECT COUNT(*) FROM products").fetchone()[0])

Products in SQLite: 500


## 8. Query expansion + metadata filters

Users may say **earphones**, **earbuds**, **TWS**, or **wireless buds**. Each query gets at least two expanded variants before retrieval. Filters are inferred conservatively from the query.

In [33]:
SYNONYMS = {
    "earphones": ["earbuds", "TWS", "wireless earbuds", "in-ear headphones"],
    "earbuds": ["earphones", "TWS", "wireless buds"],
    "tws": ["earbuds", "earphones", "true wireless stereo"],
    "buds": ["earbuds", "TWS", "earphones"],
    "noise cancellation": ["ANC", "active noise cancellation"],
    "noice cancellation": ["ANC", "active noise cancellation"],
}


def extract_filters(query: str) -> Dict[str, Any]:
    q = query.lower()
    f: Dict[str, Any] = {}
    for cat in categories:
        if cat.lower() in q:
            f["category"] = cat
    if any(w in q for w in ["earbud", "earphone", "tws", "buds"]):
        f["category"] = "Electronics"
        f["product_type"] = "Wireless Earbuds"
    m = re.search(r"(?:under|below|less than|up to)\s*[₹rs.]*\s*(\d{3,5})", q)
    if m: f["max_price"] = float(m.group(1).replace(",", ""))
    m = re.search(r"(?:above|over|more than)\s*[₹rs.]*\s*(\d{3,5})", q)
    if m: f["min_price"] = float(m.group(1).replace(",", ""))
    m = re.search(r"(\d+)\s*hours?", q)
    if m: f["min_battery_hours"] = float(m.group(1))
    return f


def expand_query(query: str) -> List[str]:
    variants = [query]
    lower = query.lower()
    additions = []
    for term, reps in SYNONYMS.items():
        if term in lower:
            additions.extend(reps[:3])
    if additions:
        variants.append(query + " " + " ".join(additions[:3]))
        variants.append("e-commerce product search " + " ".join(additions[:3]) + " " + query)
    else:
        variants.append(query + " product catalogue ShopSmart India")
        variants.append("ShopSmart listing specification policy " + query)
    return list(dict.fromkeys(variants[:3]))

print(expand_query("best wireless earphones under ₹2000 with noise cancellation"))
print(extract_filters("best wireless earphones under ₹2000 with 20 hours battery"))

['best wireless earphones under ₹2000 with noise cancellation', 'best wireless earphones under ₹2000 with noise cancellation earbuds TWS wireless earbuds', 'e-commerce product search earbuds TWS wireless earbuds best wireless earphones under ₹2000 with noise cancellation']
{'category': 'Electronics', 'product_type': 'Wireless Earbuds', 'max_price': 2000.0, 'min_battery_hours': 20.0}


## 9. BM25 + FAISS hybrid retrieval (stretch goal)

Dense retrieval is good for meaning; BM25 is good at exact product names and IDs. We combine them with reciprocal-rank fusion, then apply product filters.

In [34]:
from rank_bm25 import BM25Okapi

class HybridRetriever:
    def __init__(self, store: EmbeddingStore):
        self.store = store
        self.tokens = [re.findall(r"\w+", c.text.lower()) for c in store.chunks]
        self.bm25 = BM25Okapi(self.tokens)

    def _candidate_ok(self, c: DocumentChunk, f: Dict[str, Any]) -> bool:
        if c.doc_type == "catalogue":
            if f.get("category") and c.metadata.get("category") != f["category"]: return False
            if f.get("min_price") is not None and c.metadata.get("price", 0) < f["min_price"]: return False
            if f.get("max_price") is not None and c.metadata.get("price", 1e9) > f["max_price"]: return False
            if f.get("product_type") and c.metadata.get("product_type") != f["product_type"]: return False
            if f.get("min_battery_hours") is not None:
                battery = re.search(r"Battery life: ([0-9.]+)", c.text)
                if battery and float(battery.group(1)) < f["min_battery_hours"]: return False
        return True

    def retrieve(self, query: str, top_k: int = 8, use_expansion=True, hybrid=True):
        variants = expand_query(query) if use_expansion else [query]
        filters = extract_filters(query)
        scores = defaultdict(float)
        for q in variants:
            if self.store.index is not None and self.store.client:
                q_emb = self.store.embed_query(q).reshape(1, -1).astype("float32")
                D, I = self.store.index.search(q_emb, min(len(self.store.chunks), max(top_k * 4, 20)))
                for rank, idx in enumerate(I[0]):
                    idx = int(idx)
                    if idx < 0: continue
                    c = self.store.chunks[idx]
                    if not self._candidate_ok(c, filters): continue
                    scores[idx] += 1.0 / (60 + rank + 1)
            if hybrid or self.store.index is None:
                bm = self.bm25.get_scores(re.findall(r"\w+", q.lower()))
                ranked = np.argsort(bm)[::-1][:max(top_k * 4, 20)]
                for rank, idx in enumerate(ranked):
                    idx = int(idx)
                    c = self.store.chunks[idx]
                    if not self._candidate_ok(c, filters): continue
                    scores[idx] += 1.0 / (60 + rank + 1)
        ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
        return [(self.store.chunks[idx], float(score)) for idx, score in ranked]

retriever = HybridRetriever(smart_store)


## 10. Retrieval inspection across all five query types

The retrieval output is deliberately shown before generation. This makes it easy to diagnose whether a failure comes from chunking/retrieval or from the language model.

In [35]:
def show_retrieval(query: str, top_k: int = 6):
    print("QUERY:", query)
    results = retriever.retrieve(query, top_k=top_k)
    for rank, (c, score) in enumerate(results, 1):
        print(f"\n[{rank}] score={score:.4f} | {c.source} | {c.section} | type={c.doc_type}")
        print(c.text[:900].replace("\n", " | "))
    return results

five_queries = [
    "What is the battery life of the boAt Airdopes 141 earbuds?",
    "Can I return a fashion item that I bought on sale during the Big Billion Days event?",
    "Which wireless earbuds under ₹3000 have the best noise cancellation and at least 20 hours battery life?",
    "What are the image requirements for listing a product in the Electronics category on ShopSmart India?",
    "If I buy a smartwatch from a third-party seller and it stops working after 45 days, what are my options?",
]
for q in five_queries:
    show_retrieval(q, top_k=6)
    print("\n" + "="*100 + "\n")

QUERY: What is the battery life of the boAt Airdopes 141 earbuds?

[1] score=0.0984 | product_catalogue.csv | P1001 | type=catalogue
Product ID: P1001 | Name: boAt Airdopes 141 | Brand: boAt | Category: Electronics | Product type: Wireless Earbuds | Price: ₹1499 | Rating: 4.2/5 | Battery life: 42.0 hours | ANC: No | Warranty: 12 months | Return window: 7 days | Seller ID: S001 | Verified seller: Yes | Tags: earbuds, earphones, TWS, in-ear, wireless

[2] score=0.0940 | product_catalogue.csv | P1101 | type=catalogue
Product ID: P1101 | Name: boAt Wireless Earbuds 97 | Brand: boAt | Category: Electronics | Product type: Wireless Earbuds | Price: ₹2999 | Rating: 3.9/5 | Battery life: 43.0 hours | ANC: No | Warranty: 12 months | Return window: 7 days | Seller ID: S002 | Verified seller: Yes | Tags: wireless earbuds, boat, earbuds, earphones, TWS

[3] score=0.0921 | product_catalogue.csv | P1081 | type=catalogue
Product ID: P1081 | Name: boAt Wireless Earbuds 77 | Brand: boAt | Category: Ele

## 11. Citation Formatting + Grounded Generation

Retrieved context is presented with `[SRC: source | section_or_product_id]` citation markers. The generation prompt restricts answers to the supplied context and requires citations. When no OpenAI key is configured, the notebook uses a deterministic extractive answer path so the processing cells remain executable without external generation.

In [36]:
def format_context(results: List[Tuple[DocumentChunk, float]]) -> str:
    blocks = []
    for c, score in results:
        source_key = c.metadata.get("product_id", c.section)
        blocks.append(f"[SRC: {c.source} | {source_key}]\n{c.text}")
    return "\n\n".join(blocks)

PERSONA_PROMPTS = {
    "Shopper": """You are CatalogueIQ for shoppers. Answer only from the supplied ShopSmart context. Compare products using only retrieved facts. Never invent specifications, warranty terms, sellers, stock, or policy rules. Cite factual claims using the supplied [SRC: ...] markers. If the context is insufficient, say so clearly.""",
    "Seller": """You are CatalogueIQ for ShopSmart sellers. Answer only from the supplied ShopSmart context. Focus on listing, image, pricing, category, compliance, and seller-policy requirements. Never invent requirements. Cite factual claims using [SRC: ...].""",
    "Support Agent": """You are CatalogueIQ for internal support agents. Answer only from the supplied ShopSmart context. For multi-hop questions, connect relevant policy sections explicitly. Do not create exceptions not present in the context. Cite factual claims using [SRC: ...].""",
}


def extractive_answer(query: str, results: List[Tuple[DocumentChunk, float]]) -> str:
    q_words = set(re.findall(r"\w+", query.lower())) - ENGLISH_STOP_WORDS
    candidates = []
    for c, score in results:
        for sent in re.split(r"(?<=[.!?])\s+", c.text):
            overlap = len(q_words & set(re.findall(r"\w+", sent.lower())))
            if overlap:
                candidates.append((overlap + score, sent.strip(), c))
    candidates.sort(key=lambda x: x[0], reverse=True)
    chosen = []
    seen = set()
    for _, sent, c in candidates:
        key = sent.lower()
        if key in seen: continue
        seen.add(key)
        chosen.append(f"{sent} [SRC: {c.source} | {c.metadata.get('product_id', c.section)}]")
        if len(chosen) >= 4: break
    return " ".join(chosen) if chosen else "I could not verify the answer from the ShopSmart India knowledge base."


def generate_answer(query: str, results: List[Tuple[DocumentChunk, float]], persona: str = "Shopper") -> str:
    if not results:
        return "I could not verify the answer from the ShopSmart India knowledge base."
    if not OPENAI_API_KEY:
        return extractive_answer(query, results)
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    context = format_context(results)
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": PERSONA_PROMPTS[persona]},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {query}\n\nAnswer only from the context and include source citations."},
        ],
    )
    answer = response.choices[0].message.content.strip()
    if "[SRC:" not in answer:
        answer += "\n\nSources: " + ", ".join(f"[SRC: {c.source} | {c.metadata.get('product_id', c.section)}]" for c, _ in results[:3])
    return answer


## 12. Comparative recommendation helper

Pure vector retrieval is not enough for comparisons. The stretch implementation uses SQLite for numeric constraints and then ranks the eligible products by ANC, battery, rating, and price. This is transparent and easy to explain in a panel.

In [37]:
def recommend_products(query: str, top_n: int = 5) -> pd.DataFrame:
    f = extract_filters(query)
    where = ["1=1"]; args = []
    if f.get("category"): where.append("category=?"); args.append(f["category"])
    if f.get("product_type"): where.append("product_type=?"); args.append(f["product_type"])
    if f.get("max_price") is not None: where.append("price<=?"); args.append(f["max_price"])
    if f.get("min_price") is not None: where.append("price>=?"); args.append(f["min_price"])
    sql = f"SELECT * FROM products WHERE {' AND '.join(where)}"
    df = pd.read_sql_query(sql, con, params=args)
    if "noise" in query.lower() or "anc" in query.lower():
        df["anc_score"] = df["anc"].astype(int) * 2
    else:
        df["anc_score"] = 0
    if f.get("min_battery_hours") is not None and "battery_hours" in df.columns:
        df = df[(df["battery_hours"].fillna(0) >= f["min_battery_hours"])]
    df["battery_score"] = df["battery_hours"].fillna(0) / max(df["battery_hours"].fillna(1).max(), 1)
    df["rating_score"] = df["rating"] / 5
    df["price_score"] = 1 - (df["price"] - df["price"].min()) / max(df["price"].max() - df["price"].min(), 1)
    df["recommendation_score"] = df["anc_score"] + 2*df["battery_score"] + 2*df["rating_score"] + df["price_score"]
    return df.sort_values("recommendation_score", ascending=False).head(top_n)


## 13. Five-turn conversational memory + personalization

The memory stores the conversation for the current session. A stated brand preference is carried into subsequent product recommendations.

In [38]:
@dataclass
class ConversationMemory:
    turns: List[Dict[str, str]] = field(default_factory=list)
    max_turns: int = 5
    preference: Dict[str, str] = field(default_factory=dict)

    def add(self, user: str, assistant: str):
        self.turns.append({"user": user, "assistant": assistant})
        self.turns = self.turns[-self.max_turns:]

    def context_text(self) -> str:
        return "\n".join(f"User: {t['user']}\nAssistant: {t['assistant']}" for t in self.turns)

    def update_preferences(self, text: str):
        m = re.search(r"prefer (?:the )?([A-Za-z0-9& ]+?)(?: over | rather than )", text, re.I)
        if m:
            self.preference["preferred_brand"] = m.group(1).strip()
        m2 = re.search(r"I prefer ([A-Za-z0-9& ]+)$", text, re.I)
        if m2:
            self.preference["preferred_brand"] = m2.group(1).strip()

memory = ConversationMemory()

def answer_with_memory(query: str, persona: str = "Shopper") -> str:
    memory.update_preferences(query)
    effective_query = query
    if memory.preference.get("preferred_brand"):
        effective_query += f" Prefer brand {memory.preference['preferred_brand']}."
    results = retriever.retrieve(effective_query, top_k=8)
    if any(w in query.lower() for w in ["best", "recommend", "which", "compare"]):
        recs = recommend_products(effective_query)
        if not recs.empty and memory.preference.get("preferred_brand"):
            brand = memory.preference["preferred_brand"].lower()
            preferred = recs[recs.brand.str.lower() == brand]
            recs = pd.concat([preferred, recs]).drop_duplicates("product_id").head(5)
        if not recs.empty:
            summary = recs[["product_id","name","brand","price","rating","battery_hours","anc"]].to_dict("records")
            answer = f"Top candidates from the filtered product index: {json.dumps(summary, default=str)}"
            answer += " [SRC: product_catalogue.csv | product records]"
        else:
            answer = generate_answer(query, results, persona)
    else:
        answer = generate_answer(query, results, persona)
    memory.add(query, answer)
    return answer


## 14. Three-persona UI (Should Have)

The UI is intentionally simple. The retrieval and evidence panel are visible for evaluation so assessors can inspect what the model actually saw.

In [1]:
import gradio as gr

def chat_fn(message, persona):
    if not message.strip():
        return "Please enter a question.", ""

    results = retriever.retrieve(
        message,
        top_k=8,
        use_expansion=True,
        hybrid=True
    )

    answer = generate_answer(message, results, persona)
    evidence = format_context(results)

    return answer, evidence


with gr.Blocks(title="CatalogueIQ") as demo:

    gr.Markdown("# CatalogueIQ — ShopSmart India")
    gr.Markdown("AI-powered product intelligence assistant")

    persona = gr.Dropdown(
        choices=["Shopper", "Seller", "Support Agent"],
        value="Shopper",
        label="Persona"
    )

    query = gr.Textbox(
        label="Ask CatalogueIQ",
        placeholder="e.g. Which wireless earbuds under ₹3000 have ANC and 20+ hours battery?",
        lines=2
    )

    btn = gr.Button("Ask")

    answer = gr.Markdown(
        label="Answer"
    )

    evidence = gr.Textbox(
        label="Retrieved Evidence",
        lines=16
    )

    btn.click(
        fn=chat_fn,
        inputs=[query, persona],
        outputs=[answer, evidence]
    )


demo.launch(share=True)

c:\data\Python-Trainings\Python-Trainings\Frontier-Engineer\week1\cap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\2531813\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\2531813\.cache\huggingface\gradio\frpc


## 15. Groundedness / out-of-catalogue handling

A good RAG system should not confidently answer a question about a product it has never seen. We use a similarity threshold and explicit insufficiency language rather than guessing.

In [40]:
def grounded_query(query: str, persona: str = "Shopper", top_k: int = 8, min_score: float = 0.18):
    results = retriever.retrieve(query, top_k=top_k)
    if not results or results[0][1] < min_score:
        return "I cannot verify that product or fact from the ShopSmart India knowledge base. I will not invent a product specification or policy rule."
    return generate_answer(query, results, persona)


## 16. Evaluation dataset: 20 questions

The test set is balanced exactly as requested: four questions for each query type, including four comparative questions and four multi-hop questions.

In [41]:
eval_questions = pd.DataFrame([
    # 4 factual
    {"id":1,"type":"Product factual lookup","question":"What is the battery life of the boAt Airdopes 141 earbuds?","answer":"42 hours"},
    {"id":2,"type":"Product factual lookup","question":"Does the Sony WF-C700N support ANC?","answer":"Yes"},
    {"id":3,"type":"Product factual lookup","question":"What is the warranty for the OnePlus Nord Buds 2?","answer":"12 months"},
    {"id":4,"type":"Product factual lookup","question":"Is the seller for boAt Airdopes 141 verified?","answer":"Yes"},
    # 4 policy
    {"id":5,"type":"Policy & eligibility","question":"Can I return a fashion item bought on sale during Big Billion Days?","answer":"Yes, the normal 10-day Fashion window applies unless the listing is explicitly non-returnable."},
    {"id":6,"type":"Policy & eligibility","question":"Are personalised products eligible for change-of-mind returns?","answer":"No"},
    {"id":7,"type":"Policy & eligibility","question":"How long after inspection is a refund generally initiated?","answer":"Within 2 business days"},
    {"id":8,"type":"Policy & eligibility","question":"How long can bank/card processing take after refund initiation?","answer":"3-7 additional business days"},
    # 4 comparative
    {"id":9,"type":"Comparative recommendation","question":"Which wireless earbuds under ₹3000 have ANC and at least 20 hours battery life?","answer":"Multiple qualifying products; Sony WF-C700N and OnePlus Nord Buds 2 and realme Buds T300 are valid examples."},
    {"id":10,"type":"Comparative recommendation","question":"Which earbuds under ₹2000 have ANC and the best battery life?","answer":"realme Buds T300 qualifies with ANC and 40 hours; other qualifying products should be compared from catalogue."},
    {"id":11,"type":"Comparative recommendation","question":"Which wireless earbuds under ₹2500 have the highest rating among ANC models?","answer":"Compare qualifying ANC earbuds using catalogue rating."},
    {"id":12,"type":"Comparative recommendation","question":"Which electronics products under ₹3000 have the strongest combination of rating and warranty?","answer":"Use catalogue fields for rating and warranty to rank eligible products."},
    # 4 seller
    {"id":13,"type":"Seller policy lookup","question":"What images are required for an Electronics listing?","answer":"At least 3 clear images: primary/front, angled or side view, and packaging/accessories view."},
    {"id":14,"type":"Seller policy lookup","question":"Can Electronics sellers upload screenshots of competitor listings?","answer":"No"},
    {"id":15,"type":"Seller policy lookup","question":"What must every product listing include?","answer":"Name, brand, category, key specs, price, stock, warranty, return eligibility, and seller ID metadata."},
    {"id":16,"type":"Seller policy lookup","question":"What should sellers verify before publishing?","answer":"Authenticity, required brand authorisation, accurate specifications, safe packaging, tax information, warranty details, and compliant returns."},
    # 4 multi-hop
    {"id":17,"type":"Multi-hop reasoning","question":"If I buy a smartwatch from a third-party seller and it stops working after 45 days, what are my options?","answer":"The standard return period is over; check the product warranty and use the warranty path if still covered. Buyer Protection may help for eligible seller/order disputes."},
    {"id":18,"type":"Multi-hop reasoning","question":"If a third-party seller sends a damaged smartwatch after the return window, what route can I use?","answer":"Buyer Protection may provide a dispute route for damaged items; warranty is separate and applies to manufacturing defects."},
    {"id":19,"type":"Multi-hop reasoning","question":"A sale-priced fashion product is defective after delivery. Can I seek help even if the promotion says non-returnable?","answer":"Damaged or incorrect deliveries are handled through Buyer Protection; a sale price alone does not remove normal returns."},
    {"id":20,"type":"Multi-hop reasoning","question":"A smartwatch warranty is still active but the seller is unresponsive. What should I do?","answer":"Use the warranty path with the invoice and defect details; seller responsibility and Buyer Protection can provide support for eligible disputes."},
])
print(eval_questions.groupby("type").size())

type
Comparative recommendation    4
Multi-hop reasoning           4
Policy & eligibility          4
Product factual lookup        4
Seller policy lookup          4
dtype: int64


## 17. Baseline evaluation + targeted improvement

**Baseline:** dense FAISS only, fewer retrieved chunks, no hybrid fusion.

**Improvement:** query expansion + BM25/FAISS fusion + deeper top-k + SQLite filtering for comparative recommendations.

The notebook measures retrieval hit-rate against the expected answer keywords before running RAGAS. This makes the engineering delta visible even when an external evaluator key is unavailable.

In [42]:
def baseline_retrieve(query: str, top_k: int = 3):
    if smart_store.index is not None and smart_store.client:
        q_emb = smart_store.embed_query(query).reshape(1, -1).astype("float32")
        D, I = smart_store.index.search(q_emb, top_k)
        return [(smart_store.chunks[int(idx)], float(D[0][rank])) for rank, idx in enumerate(I[0]) if idx >= 0]
    bm = retriever.bm25.get_scores(re.findall(r"\w+", query.lower()))
    ranked = np.argsort(bm)[::-1][:top_k]
    return [(retriever.store.chunks[int(idx)], float(bm[int(idx)])) for idx in ranked]


def answer_keyword_hit(query: str, expected: str, results: List[Tuple[DocumentChunk,float]]) -> bool:
    context = " ".join(c.text.lower() for c,_ in results)
    terms = [t for t in re.findall(r"\b[a-z0-9]+\b", expected.lower()) if len(t) > 2]
    hits = sum(t in context for t in terms)
    return hits >= max(1, math.ceil(len(terms) * 0.45))


def retrieval_eval(retrieve_fn):
    rows = []
    for _, row in eval_questions.iterrows():
        res = retrieve_fn(row.question)
        rows.append({"id": row.id, "type": row.type, "hit": answer_keyword_hit(row.question, row.answer, res)})
    df = pd.DataFrame(rows)
    return df, df.hit.mean(), df.groupby("type").hit.mean()

base_df, base_hit, base_by_type = retrieval_eval(lambda q: baseline_retrieve(q, 3))
impr_df, impr_hit, impr_by_type = retrieval_eval(lambda q: retriever.retrieve(q, 8, use_expansion=True, hybrid=True))

print("Baseline retrieval hit rate:", round(base_hit, 3))
print(base_by_type.round(3))
print("\nImproved retrieval hit rate:", round(impr_hit, 3))
print(impr_by_type.round(3))
print("\nDelta:", round(impr_hit - base_hit, 3))



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Baseline retrieval hit rate: 0.8
type
Comparative recommendation    0.50
Multi-hop reasoning           1.00
Policy & eligibility          0.75
Product factual lookup        1.00
Seller policy lookup          0.75
Name: hit, dtype: float64

Improved retrieval hit rate: 0.85
type
Comparative recommendation    0.75
Multi-hop reasoning           1.00
Policy & eligibility          0.75
Product factual lookup        1.00
Seller policy lookup          0.75
Name: hit, dtype: float64

Delta: 0.05


## 18. RAGAS evaluation suite

The capstone requires RAGAS scores for **faithfulness, answer relevancy, and context precision** across all 20 questions. The function below is version-tolerant and uses OpenAI only when `OPENAI_API_KEY` is present. Without a key, the notebook still builds the complete RAGAS input dataset and clearly reports that the external evaluator must be run after adding a key.

In [45]:
def build_ragas_dataframe(eval_df, use_improved=False):
    rows = []

    for _, row in eval_df.iterrows():
        query = row["question"]

        if use_improved:
            results = retriever.retrieve(
                query,
                top_k=8,
                use_expansion=True,
                hybrid=True
            )
        else:
            results = baseline_retrieve(query, top_k=3)

        context = [c.text for c, _ in results]
        response = generate_answer(query, results, "Shopper")

        rows.append({
            "user_input": query,
            "retrieved_contexts": context,
            "response": response,
            "reference": row["answer"],
            "type": row["type"]
        })

    return pd.DataFrame(rows)


ragas_baseline_df = build_ragas_dataframe(
    eval_questions,
    use_improved=False
)

ragas_improved_df = build_ragas_dataframe(
    eval_questions,
    use_improved=True
)

print("Baseline RAGAS dataset:", ragas_baseline_df.shape)
print("Improved RAGAS dataset:", ragas_improved_df.shape)

Baseline RAGAS dataset: (20, 5)
Improved RAGAS dataset: (20, 5)


In [46]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision
)

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

llm = ChatOpenAI(
    model=GENERATION_MODEL,
    temperature=0,
    api_key=OPENAI_API_KEY
)

ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
ragas_llm = LangchainLLMWrapper(llm)

ragas_metrics = [
    faithfulness,
    answer_relevancy,
    context_precision
]

def run_ragas(eval_df):
    dataset = Dataset.from_pandas(
        eval_df[[
            "user_input",
            "response",
            "retrieved_contexts",
            "reference"
        ]],
        preserve_index=False
    )

    result = evaluate(
        dataset=dataset,
        metrics=ragas_metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False
    )

    return (
        result.to_pandas()
        if hasattr(result, "to_pandas")
        else pd.DataFrame(result)
    )

ragas_baseline_scores = run_ragas(ragas_baseline_df)
ragas_improved_scores = run_ragas(ragas_improved_df)

print("Baseline RAGAS results:")
display(ragas_baseline_scores)

print("Improved RAGAS results:")
display(ragas_improved_scores)

C:\Users\2531813\AppData\Local\Temp\ipykernel_2860\2517392770.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\2531813\AppData\Local\Temp\ipykernel_2860\2517392770.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\2531813\AppData\Local\Temp\ipykernel_2860\2517392770.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\2531813\AppData\Local\Temp\ipykern

Baseline RAGAS results:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,What is the battery life of the boAt Airdopes ...,[Product ID: P1001\nName: boAt Airdopes 141\nB...,The battery life of the boAt Airdopes 141 earb...,42 hours,1.000000,1.000000,1.000000
1,Does the Sony WF-C700N support ANC?,[Product ID: P1002\nName: Sony WF-C700N\nBrand...,"Yes, the Sony WF-C700N supports ANC (Active No...",Yes,0.500000,0.859674,1.000000
2,What is the warranty for the OnePlus Nord Buds 2?,[Product ID: P1004\nName: OnePlus Nord Buds 2\...,The warranty for the OnePlus Nord Buds 2 is 12...,12 months,1.000000,0.971873,1.000000
3,Is the seller for boAt Airdopes 141 verified?,[Product ID: P1001\nName: boAt Airdopes 141\nB...,"Yes, the seller for boAt Airdopes 141 is verif...",Yes,1.000000,1.000000,1.000000
4,Can I return a fashion item bought on sale dur...,[During a promotional event such as Big Billio...,"Yes, you can return a fashion item bought on s...","Yes, the normal 10-day Fashion window applies ...",1.000000,1.000000,1.000000
5,Are personalised products eligible for change-...,"[Personalised products, intimate-use products,...","No, personalised products are generally not el...",No,1.000000,1.000000,1.000000
6,How long after inspection is a refund generall...,[FAQ 22 Q: How long does a refund take after i...,Refunds are generally initiated within 2 busin...,Within 2 business days,1.000000,0.902097,0.833333
7,How long can bank/card processing take after r...,[FAQ 23 Q: How long can a card refund take to ...,Bank/card processing can take 3-7 additional b...,3-7 additional business days,1.000000,0.979848,1.000000
8,Which wireless earbuds under ₹3000 have ANC an...,[Product ID: P1005\nName: realme Buds T300\nBr...,The wireless earbuds under ₹3000 that have ANC...,Multiple qualifying products; Sony WF-C700N an...,1.000000,0.987024,0.833333
9,Which earbuds under ₹2000 have ANC and the bes...,"[For earbuds, compare price, battery hours, AN...",The context does not provide any earbuds under...,realme Buds T300 qualifies with ANC and 40 hou...,0.833333,0.000000,0.000000


Improved RAGAS results:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision
0,What is the battery life of the boAt Airdopes ...,[Product ID: P1001\nName: boAt Airdopes 141\nB...,The battery life of the boAt Airdopes 141 earb...,42 hours,1.000000,1.000000,1.000000
1,Does the Sony WF-C700N support ANC?,[Product ID: P1002\nName: Sony WF-C700N\nBrand...,"Yes, the Sony WF-C700N supports ANC (Active No...",Yes,0.500000,0.859691,1.000000
2,What is the warranty for the OnePlus Nord Buds 2?,[Product ID: P1004\nName: OnePlus Nord Buds 2\...,The warranty for the OnePlus Nord Buds 2 is 12...,12 months,1.000000,0.971873,1.000000
3,Is the seller for boAt Airdopes 141 verified?,[Product ID: P1001\nName: boAt Airdopes 141\nB...,"Yes, the seller for boAt Airdopes 141 is verif...",Yes,1.000000,1.000000,1.000000
4,Can I return a fashion item bought on sale dur...,[During a promotional event such as Big Billio...,"Yes, you can return a fashion item bought on s...","Yes, the normal 10-day Fashion window applies ...",0.750000,1.000000,1.000000
5,Are personalised products eligible for change-...,"[Personalised products, intimate-use products,...","No, personalised products are generally not el...",No,1.000000,1.000000,0.875000
6,How long after inspection is a refund generall...,[FAQ 22 Q: How long does a refund take after i...,A refund is generally initiated within 2 busin...,Within 2 business days,1.000000,0.902097,0.833333
7,How long can bank/card processing take after r...,[FAQ 23 Q: How long can a card refund take to ...,Bank/card processing can take 3-7 additional b...,3-7 additional business days,1.000000,0.979848,1.000000
8,Which wireless earbuds under ₹3000 have ANC an...,[Product ID: P1031\nName: boAt Wireless Earbud...,The following wireless earbuds under ₹3000 hav...,Multiple qualifying products; Sony WF-C700N an...,0.882353,0.906631,0.208333
9,Which earbuds under ₹2000 have ANC and the bes...,[Product ID: P1005\nName: realme Buds T300\nBr...,The earbuds under ₹2000 that have ANC and the ...,realme Buds T300 qualifies with ANC and 40 hou...,0.909091,0.986757,1.000000


## 19. Report generation

This creates the `README.md` and `evaluation_report.md` deliverables specified in the capstone. The report records the actual chunk counts and retrieval improvement measured above, and explicitly describes limitations instead of fabricating evaluator scores.

In [47]:
ragas_baseline_summary = (
    ragas_baseline_scores[
        ["faithfulness", "answer_relevancy", "context_precision"]
    ].mean().to_frame().T
)

ragas_improved_summary = (
    ragas_improved_scores[
        ["faithfulness", "answer_relevancy", "context_precision"]
    ].mean().to_frame().T
)

print("Baseline summary:")
display(ragas_baseline_summary)

print("Improved summary:")
display(ragas_improved_summary)

Baseline summary:


,faithfulness,answer_relevancy,context_precision
0,0.8675,0.837003,0.825


Improved summary:


,faithfulness,answer_relevancy,context_precision
0,0.852608,0.928085,0.838781


In [48]:
from pathlib import Path

# Save files in the current notebook folder
report_path = Path.cwd() / "evaluation_report.md"
readme_path = Path.cwd() / "README.md"

# Overall RAGAS scores
baseline_metrics = ragas_baseline_scores[
    ["faithfulness", "answer_relevancy", "context_precision"]
].mean()

improved_metrics = ragas_improved_scores[
    ["faithfulness", "answer_relevancy", "context_precision"]
].mean()

# Query-type RAGAS summary
def make_type_summary(scores, eval_df):
    df = scores.copy()

    if "type" not in df.columns:
        df["type"] = eval_df["type"].reset_index(drop=True)

    return (
        df.groupby("type")[
            ["faithfulness", "answer_relevancy", "context_precision"]
        ]
        .mean()
        .reset_index()
    )

baseline_type = make_type_summary(
    ragas_baseline_scores,
    ragas_baseline_df
)

improved_type = make_type_summary(
    ragas_improved_scores,
    ragas_improved_df
)

comparison = baseline_type.merge(
    improved_type,
    on="type",
    suffixes=("_baseline", "_improved")
)

comparison["faithfulness_delta"] = (
    comparison["faithfulness_improved"]
    - comparison["faithfulness_baseline"]
)

comparison["answer_relevancy_delta"] = (
    comparison["answer_relevancy_improved"]
    - comparison["answer_relevancy_baseline"]
)

comparison["context_precision_delta"] = (
    comparison["context_precision_improved"]
    - comparison["context_precision_baseline"]
)

comparison_report = comparison.rename(
    columns={
        "type": "Query Type",
        "faithfulness_baseline": "Baseline Faithfulness",
        "faithfulness_improved": "Improved Faithfulness",
        "faithfulness_delta": "Faithfulness Delta",
        "answer_relevancy_baseline": "Baseline Answer Relevancy",
        "answer_relevancy_improved": "Improved Answer Relevancy",
        "answer_relevancy_delta": "Answer Relevancy Delta",
        "context_precision_baseline": "Baseline Context Precision",
        "context_precision_improved": "Improved Context Precision",
        "context_precision_delta": "Context Precision Delta"
    }
)

ragas_table = comparison_report.to_markdown(
    index=False,
    floatfmt=".3f"
)

report_lines = [
    "# CatalogueIQ Evaluation Report",
    "",
    "## 1. Dataset Summary",
    "",
    "- 500 product records across 5 categories.",
    "- Categories: Electronics, Fashion, Home & Kitchen, Beauty, and Books.",
    "- Source formats: CSV, Markdown, HTML.",
    "- Product review summaries: CSV.",
    f"- Raw chunks: {len(raw_chunks)}.",
    f"- Fixed-size chunks: {len(fixed_chunks)}.",
    f"- Structure-aware chunks: {len(smart_chunks)}.",
    f"- Approximate total tokens: {approx_tokens:,}.",
    f"- Embedding model: `{EMBED_MODEL_NAME}`.",
    "- Vector store: FAISS `IndexFlatIP`.",
    "",
    "## 2. Chunking Decision",
    "",
    f"Two chunking strategies were implemented and inspected. Fixed-size "
    f"chunking produced {len(fixed_chunks)} chunks and can split related "
    "product fields across chunk boundaries.",
    "",
    f"Structure-aware chunking produced {len(smart_chunks)} chunks. Each "
    "catalogue row is kept as one semantic product chunk, while Markdown "
    "and HTML content is split around meaningful document boundaries.",
    "",
    "Structure-aware chunking was selected because it preserves relationships "
    "between product attributes such as price, battery life, warranty, seller "
    "information, and return window.",
    "",
    "## 3. Retrieval Design",
    "",
    "- Query expansion generates multiple variants for every query.",
    "- E-commerce synonyms include earphones, earbuds, TWS, and in-ear headphones.",
    "- Metadata filtering supports category, price range, product type, and battery threshold.",
    "- SQLite provides exact filtering for structured product constraints.",
    "- FAISS provides dense vector retrieval.",
    "- BM25 provides lexical retrieval for exact names and identifiers.",
    "- Reciprocal-rank fusion combines dense and lexical retrieval.",
    "- Improved retrieval depth: top-k=8.",
    "",
    "## 4. Context Formatting and Persona Prompts",
    "",
    "Retrieved chunks use `[SRC: source | section_or_product_id]` citation "
    "markers. The generation prompt instructs the model to answer only from "
    "retrieved context, avoid invented facts, cite factual claims, and state "
    "when the available evidence is insufficient.",
    "",
    "### Shopper",
    "",
    "```text",
    PERSONA_PROMPTS["Shopper"],
    "```",
    "",
    "### Seller",
    "",
    "```text",
    PERSONA_PROMPTS["Seller"],
    "```",
    "",
    "### Support Agent",
    "",
    "```text",
    PERSONA_PROMPTS["Support Agent"],
    "```",
    "",
    "## 5. Baseline vs Improvement",
    "",
    f"- Baseline retrieval hit rate: {base_hit:.3f}",
    f"- Improved retrieval hit rate: {impr_hit:.3f}",
    f"- Delta: {impr_hit - base_hit:+.3f}",
    "",
    "The baseline used dense FAISS retrieval with top-k=3 and no query "
    "expansion. The improved pipeline added query expansion, BM25/FAISS "
    "hybrid retrieval, deeper retrieval, metadata filtering, and "
    "SQLite-backed numeric constraints.",
    "",
    "## 6. RAGAS Evaluation",
    "",
    "RAGAS was executed across all 20 evaluation questions using faithfulness, "
    "answer relevancy, and context precision.",
    "",
    "### Overall Scores",
    "",
    "| Metric | Baseline | Improved | Delta |",
    "|---|---:|---:|---:|",
    f"| Faithfulness | {baseline_metrics['faithfulness']:.3f} | "
    f"{improved_metrics['faithfulness']:.3f} | "
    f"{improved_metrics['faithfulness'] - baseline_metrics['faithfulness']:+.3f} |",
    f"| Answer Relevancy | {baseline_metrics['answer_relevancy']:.3f} | "
    f"{improved_metrics['answer_relevancy']:.3f} | "
    f"{improved_metrics['answer_relevancy'] - baseline_metrics['answer_relevancy']:+.3f} |",
    f"| Context Precision | {baseline_metrics['context_precision']:.3f} | "
    f"{improved_metrics['context_precision']:.3f} | "
    f"{improved_metrics['context_precision'] - baseline_metrics['context_precision']:+.3f} |",
    "",
    "### Results by Query Type",
    "",
    ragas_table,
    "",
    "### Interpretation",
    "",
    f"The improved retrieval pipeline increased retrieval hit rate from "
    f"{base_hit:.3f} to {impr_hit:.3f}.",
    "",
    f"Answer relevancy changed from {baseline_metrics['answer_relevancy']:.3f} "
    f"to {improved_metrics['answer_relevancy']:.3f}, while context precision "
    f"changed from {baseline_metrics['context_precision']:.3f} to "
    f"{improved_metrics['context_precision']:.3f}.",
    "",
    f"Faithfulness changed from {baseline_metrics['faithfulness']:.3f} to "
    f"{improved_metrics['faithfulness']:.3f}. This indicates a trade-off: "
    "the improved retrieval pipeline provided a broader and more relevant "
    "candidate set, but the generation stage did not consistently use the "
    "additional context as faithfully as the baseline.",
    "",
    "This result is treated as an engineering finding rather than claiming "
    "that every evaluation metric improved simultaneously.",
    "",
    "## 7. What Did Not Work",
    "",
    "1. Fixed-size chunking can separate related product facts across chunks.",
    "2. Very small retrieval depth is insufficient for comparative recommendation queries.",
    "3. Dense-only retrieval is less effective for some exact product names and identifiers.",
    "",
    "## 8. Lessons Learned",
    "",
    "- Structured and unstructured sources require different chunking boundaries.",
    "- Comparative recommendation quality depends strongly on candidate-set recall.",
    "- Structured numeric constraints are better handled with database filtering.",
    "- Hybrid retrieval improves robustness for exact product matching.",
    "- More retrieved context does not automatically produce more faithful answers.",
    "",
    "## 9. Scalability Considerations",
    "",
    "With 100,000 additional products, the main pressure points would be "
    "embedding generation time, vector-index size, and retrieval latency.",
    "",
    "Batched embedding generation, persistent caching, incremental FAISS "
    "updates, and database-backed filtering would help maintain performance.",
    "",
    "## 10. Conclusion",
    "",
    "CatalogueIQ demonstrates a RAG-based product intelligence assistant "
    "over mixed structured and unstructured ShopSmart India data. The system "
    "supports product factual lookup, policy and eligibility questions, "
    "comparative recommendations, seller policy lookup, and multi-hop "
    "reasoning while providing grounded answers and source citations."
]

report = "\n".join(report_lines)

report_path.write_text(report, encoding="utf-8")

readme = "\n".join([
    "# CatalogueIQ",
    "",
    "RAG-powered Product Intelligence Assistant for the fictional ShopSmart India e-commerce marketplace.",
    "",
    "## Run",
    "",
    "1. Open `CatalogueIQ.ipynb`.",
    "2. Ensure `OPENAI_API_KEY` is available through a local `.env` file.",
    "3. Run the notebook cells from top to bottom.",
    "4. The notebook creates the knowledge base, indexing pipeline, retrieval system, RAGAS evaluation, and evaluation report.",
    "",
    "## Knowledge Base",
    "",
    "- 500 products across five categories.",
    "- Returns and refunds policy.",
    "- Promotional terms.",
    "- Seller onboarding guide.",
    "- Buyer FAQ.",
    "- Category taxonomy and attribute guides.",
    "- Product review summaries.",
    "",
    "## Architecture",
    "",
    "CSV / Markdown / HTML → loaders → structure-aware chunks → OpenAI `text-embedding-3-small` → FAISS + BM25 → query expansion → metadata filtering → SQLite filtering → grounded generation → citations.",
    "",
    "## Evaluation",
    "",
    "20 evaluation questions covering all five required query types.",
    "",
    "RAGAS metrics:",
    "- Faithfulness",
    "- Answer Relevancy",
    "- Context Precision",
    "",
    "## Security",
    "",
    "Keep `.env` out of version control. Never place the API key inside the notebook."
])

readme_path.write_text(readme, encoding="utf-8")

print("Report saved to:")
print(report_path.resolve())
print()
print("README saved to:")
print(readme_path.resolve())

Report saved to:
C:\data\Python-Trainings\Python-Trainings\Frontier-Engineer\week1\evaluation_report.md

README saved to:
C:\data\Python-Trainings\Python-Trainings\Frontier-Engineer\week1\README.md


In [ ]:
requirements = """pandas
numpy
scikit-learn
faiss-cpu
rank-bm25
beautifulsoup4
lxml
gradio
openai
python-dotenv
ragas
langchain-openai
"""
Path("requirements.txt").write_text(requirements, encoding="utf-8")
print("requirements.txt created")
